#### **Phase 1: Convert Directory Structure into YOLO Format**

In [4]:
import os
import shutil
import random
from PIL import Image

# --- CONFIGURATION ---
RAW_DATASET_DIR = "Datasets/face_database"  # Target raw folder
OUTPUT_YOLO_DIR = "Datasets/yolo_dataset"    # Structured output folder
TRAIN_SPLIT = 0.8                   # 80/20 data split proportion

# Generate structural target folders
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_YOLO_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_YOLO_DIR, split, 'labels'), exist_ok=True)

# Parse identities based on subdirectories
classes = sorted([d for d in os.listdir(RAW_DATASET_DIR) if os.path.isdir(os.path.join(RAW_DATASET_DIR, d))])
print(f"Found {len(classes)} classes to process: {classes}")

for class_id, class_name in enumerate(classes):
    class_path = os.path.join(RAW_DATASET_DIR, class_name)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    random.shuffle(images)
    split_idx = int(len(images) * TRAIN_SPLIT)
    
    for idx, img_name in enumerate(images):
        src_img_path = os.path.join(class_path, img_name)
        
        try:
            # Extract image sizing securely using Pillow
            with Image.open(src_img_path) as img:
                w, h = img.size
        except Exception:
            continue # Skip corrupted files safely
            
        # Since the image is already a cropped face directory, target the entire space 
        # YOLO coordinates format: class_id x_center y_center width height (Normalized 0-1)
        x_center, y_center = 0.5, 0.5
        norm_w, norm_h = 1.0, 1.0
        
        split = 'train' if idx < split_idx else 'val'
        
        # Copy image asset over
        dest_filename = f"{class_name}_{img_name}"
        shutil.copy(src_img_path, os.path.join(OUTPUT_YOLO_DIR, split, 'images', dest_filename))
        
        # Generate matched label coordinates file
        label_filename = f"{class_name}_{img_name.rsplit('.', 1)[0]}.txt"
        label_file_path = os.path.join(OUTPUT_YOLO_DIR, split, 'labels', label_filename)
        
        with open(label_file_path, 'w') as f:
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

# Output the data configuration file
yaml_content = f"""
path: {os.path.abspath(OUTPUT_YOLO_DIR)}
train: train/images
val: val/images

nc: {len(classes)}
names: {classes}
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("Data translation finalized! data.yaml configured successfully.")

Found 7 classes to process: ['Animesh', 'Nivan Tyagi', 'Rakesh Ranjan', 'Shrinav Tyagi', 'Sunny Singh', 'Tejrit Tyagi', 'Vimala']
Data translation finalized! data.yaml configured successfully.


#### **Phase 2: Running Live Tracker Cam Inference**

In [4]:
import cv2
from ultralytics import YOLO

# 1. Load your custom trained model weights 
# (Use the best weights generated from your training runs)
try:
    model = YOLO("runs/detect/train/weights/best.pt")
    print("Successfully loaded custom trained face model!")
except Exception:
    print("Custom weights not found yet. Defaulting to base model for test.")
    model = YOLO("yolo26n.pt") 

# 2. Launch video stream window
cap = cv2.VideoCapture(0)

# Check if camera opened correctly
if not cap.isOpened():
    print("Error: Could not open webcam. Check camera permissions or index.")
    exit()

print("Camera stream starting... Press 'q' inside the video window to exit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame from camera.")
        break

    # Initialize annotated_frame to the raw frame as a baseline fallback
    annotated_frame = frame.copy()

    # Run inference directly using predict()
    # conf=0.25 ignores weak false positives
    results = model.predict(frame, stream=True, conf=0.25)
    
    for r in results:
        # If objects are detected, plot them over our frame array
        annotated_frame = r.plot()
        
    # Display the processed frame
    cv2.imshow("YOLO26 Live Tracking Frame Loop", annotated_frame)
    
    # Using 30ms instead of 1ms gives your operating system time to draw the window elements
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Cleanup system resources properly
cap.release()
cv2.destroyAllWindows()

Custom weights not found yet. Defaulting to base model for test.
Camera stream starting... Press 'q' inside the video window to exit.

0: 480x640 1 person, 1 chair, 17.0ms
Speed: 1.2ms preprocess, 17.0ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 18.2ms
Speed: 1.2ms preprocess, 18.2ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 18.3ms
Speed: 1.1ms preprocess, 18.3ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 18.2ms
Speed: 1.2ms preprocess, 18.2ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 18.1ms
Speed: 1.0ms preprocess, 18.1ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 20.1ms
Speed: 1.2ms preprocess, 20.1ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 17.2ms
Speed: 0.5ms preprocess, 17.2ms inference, 0.3ms postprocess pe

#### **1. The Custom Training Script**

In [1]:
import os
from ultralytics import YOLO

def train_custom_model():
    print("--- Starting YOLO26 Custom Fine-Tuning Sequence ---")
    
    # 1. Load the core architecture model
    model = YOLO("yolo26n.pt") 
    
    # 2. Run the custom training cycle and explicitly define the output directory
    model.train(
        data="Datasets/yolo_dataset/data.yaml",       # Points to your custom dataset map file
        epochs=30,              # Adjust epochs based on dataset depth
        imgsz=640,              # Standard resolution profile
        device="cpu",           # Change to 0 or 'cuda' if using an NVIDIA GPU
        workers=2,              # Prevents multi-threading system locks
        
        # --- OUTPUT LOCATION ROUTING ---
        project="Datasets",     # Creates/uses a 'Datasets' folder in the current working directory
        name="train",           # Subfolder where weights and metrics will be saved (Datasets/train)
        exist_ok=True           # Overwrites or reuses the folder instead of creating 'train2', 'train3', etc.
    )
    print("Training finished! Core metrics saved directly to: Datasets/train/")


In [2]:
train_custom_model()

--- Starting YOLO26 Custom Fine-Tuning Sequence ---
Ultralytics 8.4.163 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (AMD Ryzen 9 7950X3D 16-Core Processor)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Datasets/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0

In [2]:
import cv2
from ultralytics import YOLO

# 1. Load the custom-trained weights generated from your dataset folders
try:
    # model = YOLO("runs/detect/train/weights/best.pt")
    # Update this line in your camera detection script:
    model = YOLO("/home/rakesh/runs/detect/Datasets/train/weights/best.pt")
    print("Successfully initialized custom facial dataset weights!")
    
    # Capture all available custom class integer IDs from your folder dataset
    custom_person_ids = list(model.names.keys())
    print(f"Tracking strict target index array: {custom_person_ids}")
    print(f"Recognized Identity Profiles: {list(model.names.values())}")
except Exception as e:
    print(f"CRITICAL ERROR loading custom weights: {e}")
    print("Ensure you successfully finished running the training sequence first.")
    exit()

# 2. Launch the standard system camera sequence
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Access denied or camera hardware busy.")
    exit()

print("\n--- Live Recognition Engine Online ---")
print("Press 'q' in the camera window to terminate application safely.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Dropped video stream frame instance.")
        break

    # Run inference filtered strictly to your custom class list
    # classes=custom_person_ids restricts the model from tracking external objects
    # conf=0.60 keeps it locked only to high-confidence matches
    results = model(frame, stream=True, conf=0.60, classes=custom_person_ids)
    
    for r in results:
        boxes = r.boxes
        for box in boxes:
            # Get location parameters
            x1, y1, x2, y2 = map(int, box.xyxy)
            
            # Map the exact matching ID back to the directory folder string name
            class_id = int(box.cls)
            person_name = model.names.get(class_id, "Unknown Target")
            confidence = float(box.conf) * 100
            
            # --- Graphical User Interface Render Layers ---
            # Draw sleek identity tracking box frame
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # Generate custom overlay strings using folder names
            label_text = f"{person_name} [{confidence:.1f}%]"
            
            # Apply dynamic text background banner
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - 25), (x1 + text_w + 10, y1), (0, 255, 0), -1)
            
            # Draw identity string clearly over target frame
            cv2.putText(frame, label_text, (x1 + 5, y1 - 7), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Display the final filtered workspace window
    cv2.imshow("Custom Dataset Restricted Face Identifier", frame)
    
    # Break loop safely via key control parameter
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("System resources safely unmapped.")

Successfully initialized custom facial dataset weights!
Tracking strict target index array: [0, 1, 2, 3, 4, 5, 6]
Recognized Identity Profiles: ['Animesh', 'Nivan Tyagi', 'Rakesh Ranjan', 'Shrinav Tyagi', 'Sunny Singh', 'Tejrit Tyagi', 'Vimala']

--- Live Recognition Engine Online ---
Press 'q' in the camera window to terminate application safely.


0: 480x640 (no detections), 18.3ms
Speed: 0.8ms preprocess, 18.3ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 18.4ms
Speed: 1.3ms preprocess, 18.4ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 19.2ms
Speed: 1.3ms preprocess, 19.2ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 18.2ms
Speed: 1.4ms preprocess, 18.2ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 19.3ms
Speed: 1.2ms preprocess, 19.3ms inference, 0.3ms postprocess per image at shape (